# Definitive Colab Notebook: Pilot Selection and Full-Training Launch

This notebook is the cleaned Colab workflow for the current JWST ViT project. It runs the full model-selection cycle first, then gives you a controlled path to launch the full-training run for the winner.

What this notebook does:
1. Mounts Google Drive, prepares a Colab-local work directory, and clones the repository.
2. Verifies the pilot catalog and keeps heavy training and evaluation outputs in Colab-local storage.
3. Trains or resumes the 9 pilot runs if needed.
4. Builds a frozen manual spiral split artifact with a cleaned label snapshot plus train/val/test CSVs.
5. Re-runs all 9 linear probes and all 9 fine-tune runs against the same frozen split artifact.
6. Uses the improved classifier path in the repo: minority-aware checkpoint selection, balanced sampling, validation-based checkpoint choice, and test-only final reporting.
7. Regenerates ranking JSON, CSV, and plots from the fresh reports.
8. Syncs only retained outputs back to Drive: the frozen split artifact, comparison artifacts, and the recommended model checkpoint bundle.
9. Gives you an optional full-training launch cell for the recommended winner.

This notebook assumes you already have a manual label CSV at `output/manual_labels/spiral_vs_not_spiral.csv` in Drive or in the repo output directory.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')

REPO_URL = 'https://github.com/nntran15/JWST-vision-transformer.git'
REPO_ROOT = Path('/content/JWST-vision-transformer')

WORK_ROOT = Path('/content/vision_transformer_work')
WORK_DATA_ROOT = WORK_ROOT / 'data' / 'JWST'
OUTPUT_ROOT = WORK_ROOT / 'output' / 'experiments'
COMPARISON_ROOT = WORK_ROOT / 'output' / 'comparison'
LABELS_ROOT = WORK_ROOT / 'output' / 'manual_labels'
WORK_SPLIT_ROOT = WORK_DATA_ROOT / 'splits'

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/vision-transformer')
DRIVE_DATA_ROOT = DRIVE_PROJECT_ROOT / 'data' / 'JWST'
DRIVE_EXPERIMENT_ROOT = DRIVE_PROJECT_ROOT / 'output' / 'experiments'
DRIVE_COMPARISON_ROOT = DRIVE_PROJECT_ROOT / 'output' / 'comparison'
DRIVE_LABELS_ROOT = DRIVE_PROJECT_ROOT / 'output' / 'manual_labels'
DRIVE_SPLIT_ROOT = DRIVE_DATA_ROOT / 'splits'
DRIVE_RETAINED_ROOT = DRIVE_PROJECT_ROOT / 'output' / 'retained_model_selection'

PILOT_CATALOG_DRIVE_DIR = DRIVE_DATA_ROOT / 'resized_10k_files'
MODEL_SELECTION_EVAL_SUBDIR = 'eval_manual_spiral_definitive'
SPLIT_ARTIFACT_NAME = 'manual_spiral_v1'

assert DRIVE_PROJECT_ROOT.exists(), f'Missing Drive project root: {DRIVE_PROJECT_ROOT}'

for directory in [
    WORK_ROOT,
    WORK_DATA_ROOT,
    OUTPUT_ROOT,
    COMPARISON_ROOT,
    LABELS_ROOT,
    WORK_SPLIT_ROOT,
    DRIVE_EXPERIMENT_ROOT,
    DRIVE_COMPARISON_ROOT,
    DRIVE_LABELS_ROOT,
    DRIVE_SPLIT_ROOT,
    DRIVE_RETAINED_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)

print('WORK_ROOT =', WORK_ROOT)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('COMPARISON_ROOT =', COMPARISON_ROOT)
print('DRIVE_PROJECT_ROOT =', DRIVE_PROJECT_ROOT)
print('DRIVE_COMPARISON_ROOT =', DRIVE_COMPARISON_ROOT)
print('DRIVE_RETAINED_ROOT =', DRIVE_RETAINED_ROOT)
print('PILOT_CATALOG_DRIVE_DIR =', PILOT_CATALOG_DRIVE_DIR)
print('MODEL_SELECTION_EVAL_SUBDIR =', MODEL_SELECTION_EVAL_SUBDIR)
print('SPLIT_ARTIFACT_NAME =', SPLIT_ARTIFACT_NAME)

In [ ]:
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements' / 'pytorch.txt')], check=True)

os.chdir(REPO_ROOT)
print('Python executable:', sys.executable)
print('Repo root:', REPO_ROOT)

In [ ]:
import tarfile

LOCAL_PILOT_CATALOG_DIR = WORK_DATA_ROOT / 'resized_10k_files'
tar_candidates = sorted(DRIVE_DATA_ROOT.glob('*.tar*'))

if LOCAL_PILOT_CATALOG_DIR.exists():
    CATALOG_DIR = LOCAL_PILOT_CATALOG_DIR
    print('Using local extracted catalog:', CATALOG_DIR)
elif tar_candidates:
    target_tar = tar_candidates[0]
    print('Extracting pilot catalog from:', target_tar)
    with tarfile.open(target_tar, 'r:*') as tar_handle:
        tar_handle.extractall(WORK_DATA_ROOT)
    CATALOG_DIR = LOCAL_PILOT_CATALOG_DIR if LOCAL_PILOT_CATALOG_DIR.exists() else PILOT_CATALOG_DRIVE_DIR
    print('Using extracted catalog:', CATALOG_DIR)
elif PILOT_CATALOG_DRIVE_DIR.exists():
    CATALOG_DIR = PILOT_CATALOG_DRIVE_DIR
    print('Using Drive catalog directly:', CATALOG_DIR)
else:
    raise FileNotFoundError('Could not find a pilot catalog directory or archive in Drive.')

pilot_count = len(list(CATALOG_DIR.glob('*.fits')))
print('Catalog FITS count:', pilot_count)
assert pilot_count > 0, f'No FITS files found under {CATALOG_DIR}'

## Helper Functions

The next cell defines the full Colab control surface for training, label snapshotting, evaluation reruns, ranking, recommendation, and optional full-training launch.

In [ ]:
import json
import re

import matplotlib.pyplot as plt

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

try:
    from IPython.display import Image, display
except ModuleNotFoundError:
    Image = None
    display = None

CONFIG_MAP = {
    'mae': 'configs/mae.yaml',
    'dino': 'configs/dino.yaml',
    'mae_dino': 'configs/mae_dino.yaml',
}

EXPERIMENTS = [
    ('mae', 'tiny'),
    ('mae', 'small'),
    ('mae', 'base'),
    ('dino', 'tiny'),
    ('dino', 'small'),
    ('dino', 'base'),
    ('mae_dino', 'tiny'),
    ('mae_dino', 'small'),
    ('mae_dino', 'base'),
]

DEFAULT_SPLIT_VAL_FRACTION = 0.2
DEFAULT_SPLIT_TEST_FRACTION = 0.2


def run_command(cmd, cwd=REPO_ROOT):
    print('Running:', ' '.join(str(part) for part in cmd))
    process = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end='')
    process.wait()
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, cmd)


def remove_path(path):
    path = Path(path)
    if not path.exists() and not path.is_symlink():
        return
    if path.is_dir() and not path.is_symlink():
        shutil.rmtree(path)
    else:
        path.unlink()


def copy_path(src, dst):
    src = Path(src)
    dst = Path(dst)
    if not src.exists():
        raise FileNotFoundError(f'Missing source path: {src}')
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        remove_path(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    return dst


def experiment_dir(method, vit_size, output_name=None, drive=False):
    name = output_name or f'pilot_{method}_timm_{vit_size}'
    root = DRIVE_EXPERIMENT_ROOT if drive else OUTPUT_ROOT
    return root / name


def evaluation_dir(method, vit_size, eval_subdir=MODEL_SELECTION_EVAL_SUBDIR, drive=False):
    return experiment_dir(method, vit_size, drive=drive) / eval_subdir


def split_artifact_dir(split_name=SPLIT_ARTIFACT_NAME, drive=False):
    root = DRIVE_SPLIT_ROOT if drive else WORK_SPLIT_ROOT
    return root / split_name


def retained_root(eval_subdir=MODEL_SELECTION_EVAL_SUBDIR):
    path = DRIVE_RETAINED_ROOT / eval_subdir
    path.mkdir(parents=True, exist_ok=True)
    return path


def comparison_artifact_paths(eval_subdir=MODEL_SELECTION_EVAL_SUBDIR, drive=False):
    root = DRIVE_COMPARISON_ROOT if drive else COMPARISON_ROOT
    return {
        'json': root / f'{eval_subdir}_model_ranking.json',
        'csv': root / f'{eval_subdir}_model_ranking.csv',
        'combined_plot': root / f'{eval_subdir}_model_ranking.png',
        'linear_plot': root / f'{eval_subdir}_linear_probe_ranking.png',
        'fine_plot': root / f'{eval_subdir}_fine_tune_ranking.png',
        'recommendation': root / f'{eval_subdir}_recommended_model.json',
    }


def get_downstream_method(method):
    return 'dino' if method == 'mae_dino' else method


def checkpoint_candidates(exp_dir, method):
    ckpt_root = Path(exp_dir) / 'checkpoints'
    candidates = []
    if method == 'mae_dino':
        candidates.extend([
            ckpt_root / 'dino_stage2' / 'checkpoint_best.pt',
            ckpt_root / 'dino_stage2' / 'checkpoint_latest.pt',
        ])
    candidates.extend([
        ckpt_root / 'checkpoint_best.pt',
        ckpt_root / 'checkpoint_latest.pt',
    ])
    return candidates


def checkpoint_path(method, vit_size, output_name=None):
    experiment_roots = [
        experiment_dir(method, vit_size, output_name=output_name, drive=False),
        experiment_dir(method, vit_size, output_name=output_name, drive=True),
    ]
    for exp_dir in experiment_roots:
        for candidate in checkpoint_candidates(exp_dir, method):
            if candidate.exists():
                return candidate
    raise FileNotFoundError(
        f'Missing checkpoint for {method} {vit_size} under '
        f'{experiment_dir(method, vit_size, output_name=output_name, drive=False) / "checkpoints"}'
    )


def sync_training_artifacts(method, vit_size, output_name=None):
    local_exp_dir = experiment_dir(method, vit_size, output_name=output_name, drive=False)
    drive_exp_dir = experiment_dir(method, vit_size, output_name=output_name, drive=True)
    synced_paths = []

    local_checkpoint = None
    for candidate in checkpoint_candidates(local_exp_dir, method):
        if candidate.exists():
            local_checkpoint = candidate
            break
    if local_checkpoint is None:
        raise FileNotFoundError(f'No local checkpoint found under {local_exp_dir / "checkpoints"}')

    relative_checkpoint = local_checkpoint.relative_to(local_exp_dir)
    synced_paths.append(copy_path(local_checkpoint, drive_exp_dir / relative_checkpoint))

    for relative_name in [Path('train.log'), Path('logs') / 'train.log']:
        local_path = local_exp_dir / relative_name
        if local_path.exists():
            synced_paths.append(copy_path(local_path, drive_exp_dir / relative_name))

    print('Synced retained training artifacts to', drive_exp_dir)
    return {
        'drive_experiment_dir': drive_exp_dir,
        'synced_paths': [str(path) for path in synced_paths],
    }


def run_train(method, vit_size, catalog_dir=CATALOG_DIR, max_samples=10000, output_name=None, force=False, sync_best_checkpoint=False):
    output_dir = experiment_dir(method, vit_size, output_name=output_name, drive=False)
    output_dir.mkdir(parents=True, exist_ok=True)

    if not force and any(path.exists() for path in checkpoint_candidates(output_dir, method)):
        print(f'Skipping training for {method} {vit_size}; checkpoint already exists at {output_dir}.')
        return output_dir

    cmd = [
        sys.executable,
        'scripts/train.py',
        '--config', CONFIG_MAP[method],
        '--framework', 'timm',
        '--vit_size', vit_size,
        '--catalog_dir', str(catalog_dir),
        '--output_dir', str(output_dir),
    ]
    if max_samples is not None:
        cmd.extend(['--max_samples', str(max_samples)])

    run_command(cmd)
    if sync_best_checkpoint:
        sync_training_artifacts(method, vit_size, output_name=output_name)
    return output_dir


def locate_labels_csv():
    candidates = [
        DRIVE_LABELS_ROOT / 'spiral_vs_not_spiral.csv',
        DRIVE_PROJECT_ROOT / 'output' / 'manual_labels' / 'spiral_vs_not_spiral.csv',
        REPO_ROOT / 'output' / 'manual_labels' / 'spiral_vs_not_spiral.csv',
        LABELS_ROOT / 'spiral_vs_not_spiral.csv',
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find output/manual_labels/spiral_vs_not_spiral.csv in Drive or in the repo output directory.')


def create_frozen_split_artifact(
    labels_csv,
    split_name=SPLIT_ARTIFACT_NAME,
    val_fraction=DEFAULT_SPLIT_VAL_FRACTION,
    test_fraction=DEFAULT_SPLIT_TEST_FRACTION,
    seed=42,
    overwrite=True,
):
    output_dir = split_artifact_dir(split_name, drive=False)
    if overwrite and output_dir.exists():
        remove_path(output_dir)

    cmd = [
        sys.executable,
        'scripts/1-data_preparation/create_manual_spiral_split_artifact.py',
        '--labels-csv', str(labels_csv),
        '--output-dir', str(output_dir),
        '--val-fraction', str(val_fraction),
        '--test-fraction', str(test_fraction),
        '--seed', str(seed),
    ]
    if overwrite:
        cmd.append('--overwrite')

    run_command(cmd)

    manifest_path = output_dir / 'split_manifest.json'
    manifest = json.loads(manifest_path.read_text())
    drive_dir = split_artifact_dir(split_name, drive=True)
    copy_path(output_dir, drive_dir)
    print('Synced frozen split artifact to', drive_dir)

    files = manifest['files']
    return {
        'artifact_dir': output_dir,
        'manifest_path': manifest_path,
        'drive_artifact_dir': drive_dir,
        'drive_manifest_path': drive_dir / 'split_manifest.json',
        'labels_snapshot_csv': output_dir / files['labels_snapshot_csv'],
        'train_csv': output_dir / files['train_csv'],
        'val_csv': output_dir / files['val_csv'],
        'test_csv': output_dir / files['test_csv'],
        'manifest': manifest,
    }


def run_classification(
    method,
    vit_size,
    labels_csv,
    split_artifact=None,
    linear_probe=True,
    fine_tune=False,
    eval_subdir=MODEL_SELECTION_EVAL_SUBDIR,
    selection_metric='spiral_f1',
    balanced_sampling=True,
    epochs=50,
    lr=1e-3,
    seed=42,
    clean_existing=False,
):
    output_dir = evaluation_dir(method, vit_size, eval_subdir=eval_subdir, drive=False)
    output_dir.mkdir(parents=True, exist_ok=True)

    if clean_existing and linear_probe:
        shutil.rmtree(output_dir / 'linear_probe', ignore_errors=True)
    if clean_existing and fine_tune:
        shutil.rmtree(output_dir / 'fine_tune', ignore_errors=True)

    cmd = [
        sys.executable,
        'scripts/evaluate.py',
        '--classify',
        '--labels_csv', str(labels_csv),
        '--checkpoint', str(checkpoint_path(method, vit_size)),
        '--framework', 'timm',
        '--method', get_downstream_method(method),
        '--vit_size', vit_size,
        '--output_dir', str(output_dir),
        '--classify_epochs', str(epochs),
        '--classify_lr', str(lr),
        '--classify_selection_metric', selection_metric,
        '--seed', str(seed),
    ]

    if split_artifact is not None:
        cmd.extend(['--split_artifact', str(split_artifact)])
    if balanced_sampling:
        cmd.append('--classify_balanced_sampling')
    if linear_probe:
        cmd.append('--linear_probe')
    if fine_tune:
        cmd.append('--fine_tune')

    run_command(cmd)
    return output_dir


def run_classification_matrix(
    mode,
    labels_csv,
    split_artifact=None,
    eval_subdir=MODEL_SELECTION_EVAL_SUBDIR,
    selection_metric='spiral_f1',
    balanced_sampling=True,
    epochs=50,
    lr=1e-3,
    clean_existing=False,
):
    if mode not in {'linear_probe', 'fine_tune'}:
        raise ValueError(f'Unsupported classification mode: {mode}')

    for method, vit_size in EXPERIMENTS:
        run_classification(
            method,
            vit_size,
            labels_csv=labels_csv,
            split_artifact=split_artifact,
            linear_probe=(mode == 'linear_probe'),
            fine_tune=(mode == 'fine_tune'),
            eval_subdir=eval_subdir,
            selection_metric=selection_metric,
            balanced_sampling=balanced_sampling,
            epochs=epochs,
            lr=lr,
            clean_existing=clean_existing,
        )


def normalize_metric_key(name):
    key = name.strip().lower().replace('-', '_').replace(' ', '_')
    if key == '0':
        return 'not_spiral'
    if key == '1':
        return 'spiral'
    return key


def parse_classification_report(report_path):
    metrics = {}
    confusion_rows = []
    in_confusion = False

    for line in report_path.read_text().splitlines():
        stripped = line.strip()
        if not stripped:
            continue

        if stripped.startswith('Selection metric:'):
            metrics['selection_metric'] = stripped.split(':', 1)[1].strip()
            continue
        if stripped.startswith('Selection score:'):
            metrics['selection_score'] = float(stripped.split(':', 1)[1].strip())
            continue
        if stripped.startswith('Report split:'):
            metrics['report_split'] = stripped.split(':', 1)[1].strip()
            continue
        if stripped.startswith('Best epoch:'):
            metrics['best_epoch'] = int(stripped.split(':', 1)[1].strip())
            continue
        if stripped == 'Confusion Matrix:':
            in_confusion = True
            continue

        if in_confusion:
            row = [int(value) for value in re.findall(r'-?\d+', stripped)]
            if row:
                confusion_rows.append(row)
            continue

        parts = stripped.split()
        if not parts:
            continue

        if parts[0] == 'accuracy' and len(parts) >= 3:
            metrics['accuracy'] = float(parts[-2])
            metrics['support'] = int(parts[-1])
            continue

        if parts[0] in {'macro', 'weighted'} and len(parts) >= 6 and parts[1] == 'avg':
            prefix = parts[0]
            metrics[f'{prefix}_precision'] = float(parts[2])
            metrics[f'{prefix}_recall'] = float(parts[3])
            metrics[f'{prefix}_f1'] = float(parts[4])
            metrics[f'{prefix}_support'] = int(parts[5])
            continue

        if len(parts) >= 5:
            label_name = normalize_metric_key(' '.join(parts[:-4]))
            if label_name in {'precision_recall_f1_score_support', 'macro_avg', 'weighted_avg'}:
                continue
            try:
                metrics[f'{label_name}_precision'] = float(parts[-4])
                metrics[f'{label_name}_recall'] = float(parts[-3])
                metrics[f'{label_name}_f1'] = float(parts[-2])
                metrics[f'{label_name}_support'] = int(parts[-1])
            except ValueError:
                continue

    if confusion_rows:
        metrics['confusion_matrix'] = confusion_rows

    return metrics


def collect_results(eval_subdir=MODEL_SELECTION_EVAL_SUBDIR):
    rows = []
    for method, vit_size in EXPERIMENTS:
        exp_dir = experiment_dir(method, vit_size, drive=False)
        for eval_type in ('linear_probe', 'fine_tune'):
            report_path = exp_dir / eval_subdir / eval_type / 'classification_report.txt'
            if not report_path.exists():
                continue
            row = {
                'experiment': exp_dir.name,
                'method': method,
                'vit_size': vit_size,
                'eval_type': eval_type,
                'report_path': str(report_path),
            }
            row.update(parse_classification_report(report_path))
            rows.append(row)
    return rows


def sort_rows(rows):
    rows.sort(
        key=lambda row: (
            row.get('spiral_f1', -1.0),
            row.get('macro_f1', -1.0),
            row.get('spiral_recall', -1.0),
            row.get('accuracy', -1.0),
        ),
        reverse=True,
    )
    return rows


def recommend_model(rows):
    linear_rows = sort_rows([row.copy() for row in rows if row['eval_type'] == 'linear_probe'])
    fine_rows = sort_rows([row.copy() for row in rows if row['eval_type'] == 'fine_tune'])

    best_linear = linear_rows[0] if linear_rows else None
    best_fine = fine_rows[0] if fine_rows else None

    if best_linear is None and best_fine is None:
        raise ValueError('No comparison rows were found.')
    if best_linear is None:
        return best_fine, best_linear, best_fine
    if best_fine is None:
        return best_linear, best_linear, best_fine

    fine_is_better = (
        best_fine.get('spiral_f1', 0.0) > best_linear.get('spiral_f1', 0.0)
        and best_fine.get('macro_f1', 0.0) >= best_linear.get('macro_f1', 0.0) - 0.02
    )
    recommended = best_fine if fine_is_better else best_linear
    return recommended, best_linear, best_fine


def build_comparison_artifacts(eval_subdir=MODEL_SELECTION_EVAL_SUBDIR):
    rows = collect_results(eval_subdir=eval_subdir)
    if not rows:
        raise ValueError(f'No classification reports were found under eval_subdir={eval_subdir}')

    linear_rows = sort_rows([row.copy() for row in rows if row['eval_type'] == 'linear_probe'])
    fine_rows = sort_rows([row.copy() for row in rows if row['eval_type'] == 'fine_tune'])
    recommended, best_linear, best_fine = recommend_model(rows)

    artifact_paths = comparison_artifact_paths(eval_subdir=eval_subdir, drive=False)
    artifact_paths['json'].write_text(json.dumps(rows, indent=2))

    fieldnames = sorted({key for row in rows for key in row.keys()})
    with artifact_paths['csv'].open('w', newline='') as handle:
        import csv

        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)

    def plot_rows(subset, title, out_path, palette):
        labels = [f"{row['experiment'].replace('pilot_', '')} [{row.get('support', '?')}]" for row in subset]
        spiral_f1 = [row.get('spiral_f1', 0.0) for row in subset]
        macro_f1 = [row.get('macro_f1', 0.0) for row in subset]
        y_values = list(range(len(subset)))
        fig, ax = plt.subplots(figsize=(12, max(5, 0.6 * len(subset) + 2)), constrained_layout=True)
        ax.barh([y + 0.2 for y in y_values], spiral_f1, height=0.35, label='spiral F1', color=palette[0])
        ax.barh([y - 0.2 for y in y_values], macro_f1, height=0.35, label='macro F1', color=palette[1])
        ax.set_yticks(y_values)
        ax.set_yticklabels(labels, fontsize=9)
        ax.invert_yaxis()
        ax.set_xlim(0, 1)
        ax.set_title(title)
        ax.set_xlabel('Score')
        ax.grid(True, axis='x', alpha=0.25)
        ax.legend(loc='lower right')
        fig.savefig(out_path, dpi=200, bbox_inches='tight')
        plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(18, 8), constrained_layout=True)
    for axis, subset, title, palette in [
        (axes[0], linear_rows, 'Linear Probe Ranking', ('#2b8cbe', '#7bccc4')),
        (axes[1], fine_rows, 'Fine-Tune Ranking', ('#d95f0e', '#fec44f')),
    ]:
        labels = [f"{row['experiment'].replace('pilot_', '')} [{row.get('support', '?')}]" for row in subset]
        spiral_f1 = [row.get('spiral_f1', 0.0) for row in subset]
        macro_f1 = [row.get('macro_f1', 0.0) for row in subset]
        y_values = list(range(len(subset)))
        axis.barh([y + 0.2 for y in y_values], spiral_f1, height=0.35, label='spiral F1', color=palette[0])
        axis.barh([y - 0.2 for y in y_values], macro_f1, height=0.35, label='macro F1', color=palette[1])
        axis.set_yticks(y_values)
        axis.set_yticklabels(labels, fontsize=8)
        axis.invert_yaxis()
        axis.set_xlim(0, 1)
        axis.set_title(title)
        axis.set_xlabel('Score')
        axis.grid(True, axis='x', alpha=0.25)
    axes[0].legend(loc='lower right')
    fig.suptitle(f'Manual Spiral Benchmark: {eval_subdir}')
    fig.savefig(artifact_paths['combined_plot'], dpi=200, bbox_inches='tight')
    plt.close(fig)

    plot_rows(linear_rows, 'Manual Spiral Benchmark: Linear Probe', artifact_paths['linear_plot'], ('#2b8cbe', '#7bccc4'))
    plot_rows(fine_rows, 'Manual Spiral Benchmark: Fine-Tune', artifact_paths['fine_plot'], ('#d95f0e', '#fec44f'))

    recommendation_payload = {
        'recommended_experiment': recommended['experiment'],
        'method': recommended['method'],
        'vit_size': recommended['vit_size'],
        'basis': recommended['eval_type'],
        'spiral_f1': recommended.get('spiral_f1'),
        'spiral_recall': recommended.get('spiral_recall'),
        'spiral_precision': recommended.get('spiral_precision'),
        'macro_f1': recommended.get('macro_f1'),
        'accuracy': recommended.get('accuracy'),
        'support': recommended.get('support'),
        'report_split': recommended.get('report_split'),
        'selection_metric': recommended.get('selection_metric'),
        'selection_score': recommended.get('selection_score'),
        'best_linear_experiment': best_linear['experiment'] if best_linear else None,
        'best_fine_experiment': best_fine['experiment'] if best_fine else None,
    }
    artifact_paths['recommendation'].write_text(json.dumps(recommendation_payload, indent=2))

    for name, path in artifact_paths.items():
        print('Wrote', name, '->', path)
    print('Recommended experiment:', recommendation_payload['recommended_experiment'])
    print('Recommendation basis:', recommendation_payload['basis'])
    print('Report split =', recommendation_payload['report_split'])
    print('spiral_f1 =', recommendation_payload['spiral_f1'])
    print('macro_f1 =', recommendation_payload['macro_f1'])

    if pd is not None:
        display(pd.DataFrame(sort_rows(rows.copy())))
    if Image is not None and display is not None:
        display(Image(filename=str(artifact_paths['combined_plot'])))
        display(Image(filename=str(artifact_paths['linear_plot'])))
        display(Image(filename=str(artifact_paths['fine_plot'])))

    return {
        'rows': rows,
        'best_linear': best_linear,
        'best_fine': best_fine,
        'recommended': recommendation_payload,
        'artifact_paths': {name: str(path) for name, path in artifact_paths.items()},
    }


def select_rows_for_retention(comparison_results, keep_runner_up=False):
    ranked_rows = sort_rows([row.copy() for row in comparison_results['rows']])
    recommendation = comparison_results['recommended']
    retained_rows = []

    for row in ranked_rows:
        if row['experiment'] == recommendation['recommended_experiment'] and row['eval_type'] == recommendation['basis']:
            retained_rows.append(row)
            break

    if keep_runner_up:
        for row in ranked_rows:
            if not retained_rows:
                retained_rows.append(row)
                continue
            first_row = retained_rows[0]
            if row['experiment'] == first_row['experiment'] and row['eval_type'] == first_row['eval_type']:
                continue
            retained_rows.append(row)
            break

    return retained_rows


def sync_retained_model_selection_artifacts(
    comparison_results,
    split_bundle=None,
    eval_subdir=MODEL_SELECTION_EVAL_SUBDIR,
    keep_runner_up=False,
):
    local_comparison_paths = comparison_artifact_paths(eval_subdir=eval_subdir, drive=False)
    drive_comparison_paths = comparison_artifact_paths(eval_subdir=eval_subdir, drive=True)
    synced_comparison = {}
    for name, local_path in local_comparison_paths.items():
        synced_comparison[name] = str(copy_path(local_path, drive_comparison_paths[name]))

    if split_bundle is not None:
        copy_path(split_bundle['artifact_dir'], split_bundle['drive_artifact_dir'])

    selected_rows = select_rows_for_retention(comparison_results, keep_runner_up=keep_runner_up)
    target_root = retained_root(eval_subdir=eval_subdir)
    retained_manifest = {
        'eval_subdir': eval_subdir,
        'split_artifact_dir': str(split_bundle['drive_artifact_dir']) if split_bundle is not None else None,
        'comparison_artifacts': synced_comparison,
        'retained_runs': [],
    }

    for row in selected_rows:
        run_root = target_root / row['experiment']
        eval_root = run_root / row['eval_type']
        local_eval_parent = evaluation_dir(row['method'], row['vit_size'], eval_subdir=eval_subdir, drive=False)
        local_eval_dir = local_eval_parent / row['eval_type']

        kept_files = []
        for source_path, destination_path in [
            (local_eval_parent / 'classification_split_summary.json', run_root / 'classification_split_summary.json'),
            (local_eval_dir / 'classification_report.txt', eval_root / 'classification_report.txt'),
            (local_eval_dir / 'best_classifier.pt', eval_root / 'best_classifier.pt'),
        ]:
            if source_path.exists():
                kept_files.append(str(copy_path(source_path, destination_path)))

        local_exp_dir = experiment_dir(row['method'], row['vit_size'], drive=False)
        for candidate in checkpoint_candidates(local_exp_dir, row['method']):
            if candidate.exists():
                relative_candidate = candidate.relative_to(local_exp_dir)
                kept_files.append(str(copy_path(candidate, run_root / 'pilot' / relative_candidate)))
                break

        retained_manifest['retained_runs'].append(
            {
                'experiment': row['experiment'],
                'eval_type': row['eval_type'],
                'files': kept_files,
            }
        )

    manifest_path = target_root / 'retained_manifest.json'
    manifest_path.write_text(json.dumps(retained_manifest, indent=2))
    print('Synced retained model-selection artifacts to', target_root)
    print('Wrote retained manifest ->', manifest_path)
    return retained_manifest


def locate_recommended_payload(eval_subdir=MODEL_SELECTION_EVAL_SUBDIR):
    candidates = [
        comparison_artifact_paths(eval_subdir=eval_subdir, drive=False)['recommendation'],
        comparison_artifact_paths(eval_subdir=eval_subdir, drive=True)['recommendation'],
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Could not find a recommendation payload for {eval_subdir}')


def launch_full_training(method, vit_size, catalog_dir, max_samples=None, output_name=None, force=False, sync_best_checkpoint=True):
    output_name = output_name or f'full_{method}_timm_{vit_size}'
    return run_train(
        method,
        vit_size,
        catalog_dir=catalog_dir,
        max_samples=max_samples,
        output_name=output_name,
        force=force,
        sync_best_checkpoint=sync_best_checkpoint,
    )

## Pilot Training Matrix

Run this cell if you still need the 9 pilot checkpoints or want to resume incomplete runs. Existing checkpoints are skipped by default.

In [ ]:
for method, vit_size in EXPERIMENTS:
    run_train(method, vit_size, catalog_dir=CATALOG_DIR, max_samples=10000, force=False)

## Create the Frozen Label Snapshot and Train/Val/Test Split Artifact

This cell sanitizes the manual labels by going through the repo split-artifact script, writes a locked label snapshot, creates deterministic train/val/test CSVs, and syncs that artifact package back to Drive for reuse.

In [ ]:
SOURCE_LABELS_CSV = locate_labels_csv()
SPLIT_ARTIFACT_BUNDLE = create_frozen_split_artifact(
    SOURCE_LABELS_CSV,
    split_name=SPLIT_ARTIFACT_NAME,
    val_fraction=DEFAULT_SPLIT_VAL_FRACTION,
    test_fraction=DEFAULT_SPLIT_TEST_FRACTION,
    seed=42,
    overwrite=True,
)
LOCKED_LABELS_CSV = SPLIT_ARTIFACT_BUNDLE['labels_snapshot_csv']
SPLIT_ARTIFACT_DIR = SPLIT_ARTIFACT_BUNDLE['artifact_dir']
SPLIT_MANIFEST_PATH = SPLIT_ARTIFACT_BUNDLE['manifest_path']
DRIVE_SPLIT_ARTIFACT_DIR = SPLIT_ARTIFACT_BUNDLE['drive_artifact_dir']

print('SOURCE_LABELS_CSV =', SOURCE_LABELS_CSV)
print('LOCKED_LABELS_CSV =', LOCKED_LABELS_CSV)
print('SPLIT_ARTIFACT_DIR =', SPLIT_ARTIFACT_DIR)
print('DRIVE_SPLIT_ARTIFACT_DIR =', DRIVE_SPLIT_ARTIFACT_DIR)
print(json.dumps(SPLIT_ARTIFACT_BUNDLE['manifest'], indent=2))

## Re-run the Full Linear-Probe Matrix

This uses the same frozen split artifact for every model, balanced sampling on the training split, validation-only checkpoint selection, and test-only final reporting.

In [ ]:
run_classification_matrix(
    mode='linear_probe',
    labels_csv=LOCKED_LABELS_CSV,
    split_artifact=SPLIT_ARTIFACT_DIR,
    eval_subdir=MODEL_SELECTION_EVAL_SUBDIR,
    selection_metric='spiral_f1',
    balanced_sampling=True,
    epochs=50,
    lr=1e-3,
    clean_existing=True,
)

## Re-run the Full Fine-Tune Matrix

This is the full end-to-end fine-tune rerun on the same frozen split artifact and the same improved selection settings.

In [ ]:
run_classification_matrix(
    mode='fine_tune',
    labels_csv=LOCKED_LABELS_CSV,
    split_artifact=SPLIT_ARTIFACT_DIR,
    eval_subdir=MODEL_SELECTION_EVAL_SUBDIR,
    selection_metric='spiral_f1',
    balanced_sampling=True,
    epochs=50,
    lr=1e-3,
    clean_existing=True,
)

## Regenerate Rankings, Plots, Recommendation, and Retained Drive Outputs

This cell rebuilds the local JSON, CSV, and plots from the fresh reports, then syncs only the retained artifacts back to Drive.

In [ ]:
comparison_results = build_comparison_artifacts(eval_subdir=MODEL_SELECTION_EVAL_SUBDIR)
retained_manifest = sync_retained_model_selection_artifacts(
    comparison_results,
    split_bundle=SPLIT_ARTIFACT_BUNDLE,
    eval_subdir=MODEL_SELECTION_EVAL_SUBDIR,
    keep_runner_up=False,
)
{
    'recommended': comparison_results['recommended'],
    'retained_manifest': retained_manifest,
}

## Optional: Launch the Full Training Run for the Winner

After you inspect the regenerated plots and retained recommendation payload, use this cell to launch the scaled-up run. The full training work stays in Colab-local storage while running, and the retained best checkpoint is copied back to Drive.

In [ ]:
recommended_payload_path = locate_recommended_payload(MODEL_SELECTION_EVAL_SUBDIR)
recommended_payload = json.loads(recommended_payload_path.read_text())
BEST_METHOD = recommended_payload['method']
BEST_SIZE = recommended_payload['vit_size']
FULL_TRAIN_CATALOG_DIR = CATALOG_DIR
FULL_TRAIN_MAX_SAMPLES = None
FULL_TRAIN_OUTPUT_NAME = f'full_{BEST_METHOD}_timm_{BEST_SIZE}'

print('RECOMMENDATION_PAYLOAD_PATH =', recommended_payload_path)
print('BEST_METHOD =', BEST_METHOD)
print('BEST_SIZE =', BEST_SIZE)
print('FULL_TRAIN_CATALOG_DIR =', FULL_TRAIN_CATALOG_DIR)
print('FULL_TRAIN_MAX_SAMPLES =', FULL_TRAIN_MAX_SAMPLES)
print('FULL_TRAIN_OUTPUT_NAME =', FULL_TRAIN_OUTPUT_NAME)

RUN_FULL_TRAINING = False
if RUN_FULL_TRAINING:
    full_training_output_dir = launch_full_training(
        BEST_METHOD,
        BEST_SIZE,
        catalog_dir=FULL_TRAIN_CATALOG_DIR,
        max_samples=FULL_TRAIN_MAX_SAMPLES,
        output_name=FULL_TRAIN_OUTPUT_NAME,
        force=False,
        sync_best_checkpoint=True,
    )
    print('Full training local output =', full_training_output_dir)
    print('Full training retained Drive dir =', experiment_dir(BEST_METHOD, BEST_SIZE, output_name=FULL_TRAIN_OUTPUT_NAME, drive=True))
else:
    print('Set RUN_FULL_TRAINING = True after you inspect the recommendation and want to start the scaled-up run.')